# Explore post-processed Amazon Reviews 2023 data

Sanity-checks on the output of `02_preprocess_amazon2023.ipynb`: split sizes, id consistency, interaction distributions, and a peek at the simulator jsonl.

# 0. Import

In [ ]:
import os
import json

import pandas as pd
from local_package.config.data import AMAZON_PROCESSED_DIR

# 1. Configuration

In [2]:
# --- config: point this at the same OUTPUT_DIR / DATA_DIR used in notebook 02 ---
CATEGORY = "All_Beauty"

DATA_DIR = AMAZON_PROCESSED_DIR / CATEGORY
OUTPUT_DIR = DATA_DIR / "chatbot"

# 2. Load outputs

In [3]:
df_train = pd.read_csv(OUTPUT_DIR / "train.tsv")
df_valid = pd.read_csv(OUTPUT_DIR / "valid.tsv")
df_test = pd.read_csv(OUTPUT_DIR / "test.tsv")
user_history = pd.read_csv(OUTPUT_DIR / "user_history.tsv")
products = pd.read_feather(OUTPUT_DIR / "products.ftr")

In [6]:
with open(OUTPUT_DIR / "map.json") as f:
    id_maps = json.load(f)

In [7]:
simulator_files = [f for f in os.listdir(OUTPUT_DIR) if f.startswith("simulator_test_data_")]
simulator_path = OUTPUT_DIR / simulator_files[0]
simulator_data = [json.loads(line) for line in open(simulator_path)]

In [8]:
print("train:", df_train.shape)
print("valid:", df_valid.shape)
print("test:", df_test.shape)
print("user_history (train+valid):", user_history.shape)
print("products:", products.shape)
print("users in map:", len(id_maps["user"]))
print("items in map:", len(id_maps["item"]))
print("simulator records:", len(simulator_data))

train: (1492, 2)
valid: (198, 2)
test: (198, 3)
user_history (train+valid): (1690, 3)
products: (280, 6)
users in map: 198
items in map: 280
simulator records: 198


## Sanity checks

In [9]:
# leave-one-out: every user should appear exactly once in valid and once in test
valid_counts = df_valid["user_id"].value_counts()
test_counts = df_test["user_id"].value_counts()
print("users with != 1 valid row:", (valid_counts != 1).sum())
print("users with != 1 test row:", (test_counts != 1).sum())

users with != 1 valid row: 0
users with != 1 test row: 0


In [10]:
# item ids in the splits should all exist in the product table
known_ids = set(products["id"])
for name, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    missing = (~df["item_id"].isin(known_ids)).sum()
    print(f"{name}: {missing} item_ids missing from products table")

train: 0 item_ids missing from products table
valid: 0 item_ids missing from products table
test: 0 item_ids missing from products table


In [11]:
# id range should match the map sizes
print("max user_id in train:", df_train["user_id"].max(), "vs users in map:", len(id_maps["user"]))
print("max item_id in train:", df_train["item_id"].max(), "vs items in map:", len(id_maps["item"]))

max user_id in train: 198 vs users in map: 198
max item_id in train: 280 vs items in map: 280


## Interaction distributions

In [12]:
all_inter = pd.concat([df_train, df_valid, df_test])
print("interactions per user:\n", all_inter.groupby("user_id").size().describe())
print("\ninteractions per item:\n", all_inter.groupby("item_id").size().describe())

interactions per user:
 count    198.000000
mean       9.535354
std        6.463192
min        5.000000
25%        6.000000
50%        7.000000
75%       11.000000
max       54.000000
dtype: float64

interactions per item:
 count    280.000000
mean       6.742857
std        1.819336
min        5.000000
25%        5.000000
50%        6.000000
75%        8.000000
max       13.000000
dtype: float64


## Product table

In [13]:
print("visited_num stats:")
print(products["visited_num"].describe())
print("\nmost-visited products:")
print(products.sort_values("visited_num", ascending=False)[["title", "visited_num"]].head(10))

visited_num stats:
count    280.000000
mean       6.035714
std        2.189616
min        0.000000
25%        5.000000
50%        6.000000
75%        7.000000
max       12.000000
Name: visited_num, dtype: float64

most-visited products:
                                                 title  visited_num
259  NATWAG Reusable Update Makeup Remover Cloths f...           12
62   Empty Amber Glass Spray Bottles - (4 Pack) 16 ...           12
59   Nail Files 16 Pcs, Double Sided Emery Board Ma...           12
205  UV LED Nail Lamp – 128W Rapid-Curing Gel Nails...           12
98   Bouquet Garni Body Shower White Musk - Unique ...           12
145  Gel Nail Polish Set, Glitter 6 Colors Gel Poli...           11
223  Lesentia Tinted Moisturizer with SPF 31 – Tint...           11
256  LUXAZA 6 PCS Purple Eyeshadow Stick,Light Shim...           11
130  MD Complete Bright & Healthy Vitamin C+ Vitali...           11
114  LANGE HAIR Extendé Conditioning Detangler Hair...           10


In [14]:
print("category distribution:")
print(products["category"].value_counts().head(20))
print(f"\nmissing/placeholder description: {(products['description'] == 'No description').mean():.1%}")
print(f"missing price: {products['price'].isna().mean():.1%}")

category distribution:
category
Unknown    280
Name: count, dtype: int64

missing/placeholder description: 95.0%
missing price: 85.0%


## Simulator jsonl peek

In [15]:
for rec in simulator_data[:5]:
    print("history:", rec["history"][:200])
    print("target: ", rec["target"])
    print()

history: Nail Clipper Set, GIDIBII Luxury 18 in 1 Stainless; RUGGED & DAPPER Active Regimen Grooming and Skinca; Purple Hair Mask for Blonde with Keratin & Jojoba ; DeepDream Portable Electric Nail Drill - Pro
target:  Keratin Secrets Do It Yourself Home Keratin System

history: Manicure and Pedicure Nail Clipper from POWERGROOM; 35.4 Inch Extra Long Silicone Back Scrubber for Sh; Lemon Oil Towelettes - 20 Count; Sasy n Savy, Citrus Soufflè Hydrating and Nourishi
target:  So'Bio Étic | Moisturizing Fresh Gel Cream | Organic Face Moisturizer for Combination Skin | 24 hr Hydrating, Balancing & Plumping | 0.5 fl oz

history: Bouquet Garni Body Shower White Musk - Unique Frag; Follain Everybody Bar Soap | Lavender & Bergamot, ; Spanature Lavender Hand & Body Lotion Travel Size ; Spanature Olive Hand & Body Lotion Travel Si
target:  Spanature Green Tea Hand & Body Lotion Travel Size Selection, 4 Pack, Revitalizing Moisturizer for Rough and Dry Skin, Nourishing and Gentle for All Skin Types,